# Lecture 7 — Advanced Visualization & Dashboards (Real API Data + Plotly)

This notebook uses a **professional workflow**:

1. Create a reproducible project folder structure (`data/raw`, `data/clean`, `output/figures`, …)
2. Pull **real macro data** from a public API (BLS Public Data API v2)
3. Save **raw** API responses (audit trail)
4. Build a **clean, analysis-ready** dataset
5. Create **interactive Plotly** visualizations designed for **decision-makers**
6. (Optional) Outline a simple Streamlit dashboard

> **Business context:** We want a decision-ready view of inflation and labor market conditions:  
> **What is inflation doing? Is it near a benchmark (e.g., 2%)? How does unemployment move alongside it?**


In [2]:
# If Plotly is missing in this environment, install it here.
# IMPORTANT: After installation, restart the kernel (Kernel → Restart) and re-run imports.

import sys, importlib.util

if importlib.util.find_spec("plotly") is None:
    print("Plotly not found. Installing...")
    !{sys.executable} -m pip install -q plotly
else:
    print("Plotly is already installed.")


Plotly is already installed.


In [4]:
from pathlib import Path
import json
import requests
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from datetime import datetime

# Make Plotly look nice in notebooks
import plotly.io as pio
pio.renderers.default = "notebook_connected"


In [6]:
# -------------------------
# (1) Create project folders
# -------------------------
PROJECT_ROOT = Path.cwd()

DIRS = {
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_clean": PROJECT_ROOT / "data" / "clean",
    "output": PROJECT_ROOT / "output",
    "figures": PROJECT_ROOT / "output" / "figures",
    "tables": PROJECT_ROOT / "output" / "tables",
    "scripts": PROJECT_ROOT / "scripts",
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("Folders ready:")
for k, p in DIRS.items():
    print(f"  {k:10s} -> {p}")


Folders ready:
  data_raw   -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/data/raw
  data_clean -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/data/clean
  output     -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/output
  figures    -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/output/figures
  tables     -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/output/tables
  scripts    -> /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/scripts


## Helper functions

We will:
- send a POST request to the BLS time series API
- save the raw JSON response into `data/raw/`
- convert the response into a tidy DataFrame


In [8]:
def save_json(obj, path: Path):
    path.write_text(json.dumps(obj, indent=2))
    print(f"Saved raw JSON: {path}")

def bls_timeseries(series_ids, start_year=2015, end_year=2025, api_key=None):
    """Pull time series from the BLS Public Data API v2."""
    url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
    payload = {
        "seriesid": series_ids,
        "startyear": str(start_year),
        "endyear": str(end_year),
    }
    if api_key:
        payload["registrationkey"] = api_key

    r = requests.post(url, json=payload, timeout=60)
    r.raise_for_status()
    out = r.json()
    # Basic sanity check
    if out.get("status") != "REQUEST_SUCCEEDED":
        raise RuntimeError(f"BLS API request failed: {out}")
    return out

def bls_to_tidy_df(bls_json):
    """Convert BLS JSON to tidy monthly data."""
    rows = []
    for s in bls_json["Results"]["series"]:
        sid = s["seriesID"]
        for item in s["data"]:
            # Monthly periods are M01..M12 (M13 is annual avg; skip it)
            period = item.get("period", "")
            if period.startswith("M") and period != "M13":
                year = int(item["year"])
                month = int(period[1:])
                date = pd.Timestamp(year=year, month=month, day=1)
                rows.append({
                    "series_id": sid,
                    "date": date,
                    "value": float(item["value"]),
                })
    df = pd.DataFrame(rows)
    df = df.sort_values(["series_id", "date"]).reset_index(drop=True)
    return df


## Business question (write 2–4 bullets)

Imagine you are advising a policymaker or a city budget office:

- Is inflation near a benchmark (like 2%)?
- When did inflation peak, and how fast did it fall?
- How does unemployment move alongside inflation?
- What would you monitor monthly?

*(Write your answer here before running the plots.)*


## Pull real data from the BLS API

We will pull two widely-used monthly series:

- **CPI-U (All items, U.S. city average, seasonally adjusted):** `CUSR0000SA0`  
- **Unemployment rate (seasonally adjusted):** `LNS14000000`

We will then save:
- raw API response → `data/raw/`
- clean monthly dataset → `data/clean/`


In [28]:
SERIES = {
    "cpi_all_items": "CUSR0000SA0",
    "cpi_core": "CUSR0000SA0L1E",
    "unemp_rate": "LNS14000000",
}

start_year, end_year = 2015, 2025

bls_json = bls_timeseries(list(SERIES.values()), start_year=start_year, end_year=end_year)

raw_path = DIRS["data_raw"] / f"bls_timeseries_{start_year}_{end_year}.json"
save_json(bls_json, raw_path)

tidy = bls_to_tidy_df(bls_json)
tidy.head()


Saved raw JSON: /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/data/raw/bls_timeseries_2015_2025.json


,series_id,date,value
0,CUSR0000SA0,2015-01-01,234.747
1,CUSR0000SA0,2015-02-01,235.342
2,CUSR0000SA0,2015-03-01,235.976
3,CUSR0000SA0,2015-04-01,236.222
4,CUSR0000SA0,2015-05-01,237.001


## Clean + build analysis-ready dataset

We pivot to one row per month and compute:
- CPI **YoY %** (12-month percent change)
- CPI **MoM %** (1-month percent change)


In [30]:
# Map series_id -> friendly column name
id_to_name = {v: k for k, v in SERIES.items()}
tidy["series_name"] = tidy["series_id"].map(id_to_name)

# Wide monthly dataset (one row per month)
wide = (
    tidy.pivot_table(index="date", columns="series_name", values="value")
        .reset_index()
        .sort_values("date")
)

# Inflation measures: Headline (All items) and Core (Less food & energy)
wide["headline_yoy_pct"] = wide["cpi_all_items"].pct_change(12) * 100
wide["headline_mom_pct"] = wide["cpi_all_items"].pct_change(1) * 100

wide["core_yoy_pct"] = wide["cpi_core"].pct_change(12) * 100
wide["core_mom_pct"] = wide["cpi_core"].pct_change(1) * 100

# Optional: headline-core spread (great for discussion)
wide["headline_minus_core_yoy"] = wide["headline_yoy_pct"] - wide["core_yoy_pct"]

# Save clean data
clean_path = DIRS["data_clean"] / f"monthly_cpi_core_unemp_{start_year}_{end_year}.csv"
wide.to_csv(clean_path, index=False)
print(f"Saved clean data: {clean_path}")

wide.tail()


Saved clean data: /Users/SHSU/Library/CloudStorage/Dropbox/Classes/BANA4373-sp26/Lectures/Lecture_7/data/clean/monthly_cpi_core_unemp_2015_2025.csv


series_name,date,cpi_all_items,cpi_core,unemp_rate,headline_yoy_pct,headline_mom_pct,core_yoy_pct,core_mom_pct,headline_minus_core_yoy
115,2024-08-01,314.062,319.750,4.2,2.607144,0.157222,3.285096,0.253966,-0.677951
116,2024-09-01,314.732,320.732,4.1,2.426483,0.213334,3.282357,0.307115,-0.855874
117,2024-10-01,315.631,321.731,4.1,2.578844,0.285640,3.298690,0.311475,-0.719846
118,2024-11-01,316.528,322.657,4.2,2.719472,0.284193,3.285936,0.287818,-0.566463
119,2024-12-01,317.604,323.259,4.1,2.870691,0.339938,3.213334,0.186576,-0.342643


## Design mistake (from Lecture 6) — then fix it

Interactivity does **not** fix misleading design.

Below, we create a **bad chart** by truncating the y-axis to exaggerate changes.  
Then we create a **decision-ready chart** with:
- meaningful baseline
- clear labels/units
- benchmark line (2%)
- annotations for key events


In [36]:
# BAD EXAMPLE: Truncated y-axis (misleading)
df_plot = wide.dropna(subset=["core_yoy_pct"]).copy()

fig_bad = px.line(
    df_plot, x="date", y="core_yoy_pct",
    title="BAD: CPI Inflation (YoY %) with Truncated Y-axis (Misleading)"
)
# Intentionally misleading range (example)
fig_bad.update_yaxes(range=[df_plot["core_yoy_pct"].quantile(0.5), df_plot["core_yoy_pct"].max()])

fig_bad.update_layout(
    xaxis_title="Date",
    yaxis_title="YoY change (%)",
)
fig_bad.show()


In [40]:
from datetime import datetime
# BETTER EXAMPLE: Decision-ready chart
fig = px.line(
    df_plot, x="date", y="core_yoy_pct",
    title="Core Inflation (YoY %) — Decision-ready view"
)

# Benchmark line: 2% (common target reference)
fig.add_hline(y=2.0, line_dash="dash", annotation_text="Benchmark: 2%")

# Annotate COVID shock (example marker)
# Event marker (robust to pandas/plotly datetime issues)
x_event = datetime(2020, 3, 1)
fig.add_shape(
    type="line",
    x0=x_event, x1=x_event,
    y0=0, y1=1,
    xref="x", yref="paper",
    line=dict(dash="dot")
)
fig.add_annotation(
    x=x_event, y=1, xref="x", yref="paper",
    text="COVID-19 shock",
    showarrow=False,
    yanchor="bottom"
)
# Annotate the peak inflation month in sample
peak_row = df_plot.loc[df_plot["core_yoy_pct"].idxmax()]
fig.add_annotation(
    x=peak_row["date"], y=peak_row["core_yoy_pct"],
    text=f"Peak: {peak_row['core_yoy_pct']:.1f}%",
    showarrow=True, arrowhead=2
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Year-over-Year change (%)",
    hovermode="x unified",
)

fig.update_traces(
    hovertemplate="<b>%{x|%Y-%m}</b><br>%{y:.2f}%"
)

fig.show()


### Interpretation check (write 3–5 sentences)

1. Why is the truncated y-axis misleading in the first chart?  
2. What does the benchmark (2%) add for a decision-maker?  
3. What story do you see around the peak and the post-peak period?


## Compare inflation and unemployment (interactive)

A common macro discussion: **inflation vs. unemployment**.

We will:
- plot both series over time
- add hover + unified mode
- discuss what we can/cannot infer from visual correlation


In [44]:
df2 = wide.dropna(subset=["headline_yoy_pct","unemp_rate"]).copy()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df2["date"], y=df2["headline_yoy_pct"],
    mode="lines", name="CPI YoY (%)"
))

fig2.add_trace(go.Scatter(
    x=df2["date"], y=df2["unemp_rate"],
    mode="lines", name="Unemployment rate (%)",
    yaxis="y2"
))

fig2.update_layout(
    title="Inflation vs Unemployment (Two-axis view)",
    xaxis=dict(title="Date"),
    yaxis=dict(title="CPI YoY (%)"),
    yaxis2=dict(title="Unemployment rate (%)", overlaying="y", side="right"),
    hovermode="x unified",
    legend_title="Series"
)

fig2.show()


## Important: Interactive ≠ Causal

Visualizations are powerful, but they do **not** prove causality.

- Charts show *patterns* and help us explore.
- Regression helps estimate relationships while controlling for other factors.

**Write 2 sentences:**
1) One thing this figure suggests  
2) One thing it *cannot* prove


## Mini “dashboard” elements

Even without a full web app, we can make a notebook feel like a dashboard:
- KPI cards (latest values)
- a few key interactive charts
- a small table for recent months


In [46]:
latest = wide.dropna(subset=["headline_yoy_pct","unemp_rate"]).sort_values("date").iloc[-1]

kpi_df = pd.DataFrame({
    "metric": ["CPI YoY (%)", "CPI MoM (%)", "Unemployment rate (%)"],
    "value": [latest["headline_yoy_pct"], latest["headline_mom_pct"], latest["unemp_rate"]],
})

kpi_df


,metric,value
0,CPI YoY (%),2.870691
1,CPI MoM (%),0.339938
2,Unemployment rate (%),4.100000


In [48]:
# Simple KPI-style display using Plotly indicator traces
figk = go.Figure()

figk.add_trace(go.Indicator(
    mode="number",
    value=float(latest["headline_yoy_pct"]),
    title={"text": "CPI YoY (%)"},
    domain={"row": 0, "column": 0},
))

figk.add_trace(go.Indicator(
    mode="number",
    value=float(latest["headline_mom_pct"]),
    title={"text": "CPI MoM (%)"},
    domain={"row": 0, "column": 1},
))

figk.add_trace(go.Indicator(
    mode="number",
    value=float(latest["unemp_rate"]),
    title={"text": "Unemployment (%)"},
    domain={"row": 0, "column": 2},
))

figk.update_layout(
    grid={"rows": 1, "columns": 3, "pattern": "independent"},
    title=f"KPI Snapshot (latest month: {latest['date']:%Y-%m})"
)

figk.show()


In [26]:
wide.sort_values("date").tail(12)[["date","cpi_u_all_items","cpi_yoy_pct","unemp_rate"]]

series_name,date,cpi_u_all_items,cpi_yoy_pct,unemp_rate
108,2024-01-01,309.698,3.088343,3.7
109,2024-02-01,310.967,3.157074,3.9
110,2024-03-01,312.345,3.486835,3.9
111,2024-04-01,313.023,3.360795,3.9
112,2024-05-01,313.175,3.244279,3.9
113,2024-06-01,313.044,2.970258,4.1
114,2024-07-01,313.569,2.941476,4.2
115,2024-08-01,314.062,2.607144,4.2
116,2024-09-01,314.732,2.426483,4.1
117,2024-10-01,315.631,2.578844,4.1


## A 20-second dashboard checklist (decision-ready)

Before you show a dashboard to someone else:

1. **Question-first title** (what decision does this support?)
2. **Units on axes** (%, dollars, index points)
3. **Benchmark line** (target/average/threshold)
4. **Annotation** (key events)
5. **Fair comparisons** (same scales when comparing)
6. **No clutter** (start with 3–5 visuals; add filters for details)


## Optional: Streamlit demo (do not run during class unless your machine is set up)

If you want to convert this notebook workflow into a simple app:

1) Save a script `app.py`  
2) Run in terminal: `streamlit run app.py`

**Note:** Streamlit may not be installed on classroom machines; treat this as optional.


In [ ]:
# Example skeleton (not executed here)
streamlit_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/clean/monthly_cpi_unemp_2015_2025.csv", parse_dates=["date"])

st.title("Inflation + Unemployment Dashboard")

st.metric("Latest CPI YoY (%)", round(df["cpi_yoy_pct"].dropna().iloc[-1], 2))
st.metric("Latest Unemployment (%)", round(df["unemp_rate"].dropna().iloc[-1], 2))

fig = px.line(df.dropna(subset=["cpi_yoy_pct"]), x="date", y="cpi_yoy_pct", title="CPI YoY (%)")
st.plotly_chart(fig, use_container_width=True)
'''
print(streamlit_code)


## Exit ticket (2 minutes)

Write:
1) One decision this dashboard could support  
2) One improvement you would make before sharing it publicly
